# Point-treatment TMLE: did transition navigation improve the experience score?

This notebook estimates one average treatment effect from observational data with targeted
maximum likelihood estimation (TMLE). Each step shows its code, its output, and what the output
tells you. [Point-treatment TMLE](../technical-reference/point-treatment-tmle.md) gives the
parameter, the influence curve, and the algorithm.

## The applied question

A regional health plan offers adults a **standard transition-navigation protocol** when a discharge
home is ordered. The offer is a bedside plan and two scheduled contacts within 30 days. Nobody
randomized it, and discharge teams used a recorded risk process.

The program sponsor asks one question. How much would the mean 30-day transition score change if
every eligible discharge received the offer, rather than usual support? That question is the
average treatment effect (ATE), not a regression coefficient.

## What you will learn

| after this notebook you can | the step that shows it |
| --- | --- |
| say why an unadjusted difference is not a causal effect | the association step |
| write the study protocol before you fit anything | the protocol step |
| name the estimand and the assumptions that identify it | the design and identification step |
| fit TMLE with explicit learners, and read its interval | the estimation step |
| tell the ATE, the ATT, and the ATC apart | the population step |
| see what double robustness does not promise | the failure mode |
| read the diagnostics and the sensitivity analysis | Steps 9 and 10 |


## Why this method

| your situation | what this method buys | what it costs |
| --- | --- | --- |
| observational data, confounders measured | double robust point consistency: either consistent nuisance model can supply it, under positivity and regularity conditions | you must name the estimand first |
| the nuisance functions are not linear | flexible learners fit both nuisances, and the estimate stays a plug-in | a valid interval needs a product rate on the two nuisances |
| you want an interval you can report | the interval comes from the targeted influence curve | positivity must hold, and a support report cannot verify it |

A regression coefficient and an inverse-probability-weighted mean each rest on one model. TMLE
targets the outcome regression with the treatment mechanism, so it uses both. The table below
defines the five terms this notebook uses most. Each step repeats the definition where the term
first matters.

| term | plain meaning |
| --- | --- |
| estimand | the number the question asks for, written before any model is chosen |
| nuisance | a model the estimate needs but the question does not ask about. Here, the outcome regression Q and the treatment mechanism g |
| targeting | a small update to Q, weighted by g, that removes the first-order bias of the plug-in estimate |
| influence curve | each row's contribution to the estimate's error. Its variance gives the standard error |
| positivity | every kind of patient has some chance of each arm, so both arms have data to compare |


## Step 1: set up

The setup imports the learners and prints the installed `cleverly` version. Every fit below passes
its learners, fold count, and random seed explicitly, so a rerun reproduces the stored outputs.


In [1]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.2


**What this output tells you.** The stored outputs in this notebook came from the version named
above. A different version can print different numbers.


## Step 2: the data

The data come from a synthetic law with a known answer. The code renames the generator's columns to
the program's names and prints the first rows. It also prints the true values of the law.


In [2]:
from cleverly.datasets import make_nonlinear_ate

frame, truth = make_nonlinear_ate(n=3_000, seed=21)
frame = frame.rename(
    columns={
        "Y": "transition_score",
        "A": "transition_navigation",
        "W1": "discharge_risk",
        "W2": "prior_utilization",
        "W3": "medication_burden",
        "W4": "age",
    }
)
print("rows and columns:", frame.shape)
print(frame.head().round(3))
print()
print("known values of the synthetic law:")
for key in ("ey1", "ey0", "ate", "att", "atc"):
    print(f"  {key}: {truth[key]:.3f}")

rows and columns: (3000, 6)
   transition_score  transition_navigation  discharge_risk  prior_utilization  medication_burden    age
0             4.283                    0.0           0.359              1.511             -1.786  1.687
1             2.482                    1.0          -0.047             -0.800             -0.803 -1.083
2             0.832                    0.0          -0.224              0.834              0.584  0.638
3             0.699                    0.0          -1.695             -1.571              1.554  0.969
4             4.429                    1.0           2.183              1.210             -1.024  1.285

known values of the synthetic law:
  ey1: 3.669
  ey0: 1.919
  ate: 1.750
  att: 1.946
  atc: 1.586


**What this output tells you.** Each row is one discharge. `transition_navigation` is 1 for an
offer and 0 for usual support. The four baseline covariates are standardized (mean 0, SD 1), so a
negative `age` is below the average age. The transition score is in synthetic units.

The true ATE is 1.750. The effect is larger than a real navigation program would expect, so each
fit shows its behavior clearly. Every discharge is independent here, and the
[cross-fitting tutorial](cross-fitting.ipynb) adds shared navigator teams.

| feature of the law | what it means in this program |
| --- | --- |
| the four baseline covariates drive assignment and the outcome | higher-risk patients are more likely to receive an offer and report different outcomes. All four are confounders in the synthetic law |
| both nuisance functions are nonlinear | a GLM is misspecified for each one, which is the condition this page exploits |
| the effect varies with the covariates | `ate`, `att`, and `atc` differ (1.750, 1.946, and 1.586), so the estimand must be named rather than inferred |

A real program has no `truth`. Every comparison against it below is a teaching device.


## Step 3: association first

A confounder is a variable that changes both who receives the offer and the outcome. The code
compares the two arms before any adjustment. It prints the mean score and the mean of each
baseline covariate by arm.


In [3]:
covariates = ["discharge_risk", "prior_utilization", "medication_burden", "age"]
by_arm = frame.groupby("transition_navigation")[["transition_score", *covariates]].mean()
print(by_arm.round(3))
print()
print("share offered navigation:", round(float(frame["transition_navigation"].mean()), 3))
unadjusted = by_arm.loc[1.0, "transition_score"] - by_arm.loc[0.0, "transition_score"]
print(f"unadjusted difference in mean score: {unadjusted:.3f}")
print(f"population ATE:                      {truth['ate']:.3f}")

                       transition_score  discharge_risk  prior_utilization  medication_burden    age
transition_navigation                                                                               
0.0                               1.894          -0.241              0.005             -0.026 -0.018
1.0                               3.829           0.259              0.010             -0.045  0.062

share offered navigation: 0.458
unadjusted difference in mean score: 1.935
population ATE:                      1.750


**What this output tells you.** The offered patients score 1.935 points higher on average, and the
true effect is 1.750. The gap exists because the arms differ before the offer. The mean
`discharge_risk` is 0.259 among offered patients and -0.241 among the others.

The unadjusted difference mixes the effect of the offer with the effect of higher discharge risk.
An estimator must compare like with like. The next two steps state which comparison the question
needs, and which assumptions make it possible.


## Step 4: write the protocol

A `StudyProtocol` records the scientific design before any model runs. It follows target-trial
vocabulary. You name the population, the time zero, the strategies, the outcome, and how later
events are handled. The record gets a fingerprint, and every result fitted from it carries that
fingerprint.

`navigation_protocol()` in `cleverly.datasets` holds the program's protocol, so every tutorial
starts from one record. A real analysis calls `StudyProtocol(...)` with the same ten fields. The
code prints each field.

In [4]:
from cleverly.datasets import navigation_protocol

protocol = navigation_protocol()
print("\n".join(protocol.summary_lines()))

causal study protocol: schema 1; 2dd268e1f5ab29ae
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measurement and before the navigation offer
treatment strategies: ['Offer standard transition navigation', 'Provide usual discharge support']
treatment versions: ['Bedside transition plan and two scheduled navigator contacts within 30 days', 'No access to the transition-navigation offer']
outcome: Patient-reported transition score
horizon: 30 days after discharge
intercurrent-event handling: ['Use the transition score regardless of readmission', 'Analyze the offer regardless of completed contacts', 'The protocol scores death before day 30 as the worst transition score (composite strategy)']
interference unit: Individual patient
assumption rationale: ['The recorded baseline variables cover 

**What this output tells you.** The first line gives the schema version and the fingerprint
`2dd268e1f5ab29ae`. The other lines repeat each field. Read them as a checklist.

| protocol field | the question it answers for this program |
| --- | --- |
| target population and eligibility | who the effect is about |
| time zero | when follow-up starts. Here, the discharge-home order, before the offer |
| treatment strategies and versions | what "offer" and "usual support" mean in practice |
| outcome and horizon | what is measured, and when |
| intercurrent-event handling | what happens to readmission, incomplete contacts, and death before day 30 |
| interference unit | whose assignment can affect whose outcome |
| assumption rationale | why the design supports the identification assumptions |

Time zero, eligibility, and the offer coincide at the discharge-home order. Eligibility therefore
cannot depend on the offer. The protocol does not store the contrast. The typed estimand in the
next step owns it.


## Step 5: design and identification

The design says which column plays which role. The estimand says which contrast you want. The two
are separate on purpose, so a later change of estimator cannot change the question.

Identification turns a causal question into a quantity the data can estimate. The `identify` call
returns that formula, the nuisance models it needs, and the assumptions that make it causal.


In [5]:
from cleverly import ATE, CausalStudy, PointTreatment

study = CausalStudy(
    frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "prior_utilization", "medication_burden", "age"),
    ),
    protocol=protocol,
)
effect = study.identify(ATE(reference=0))

print(effect.summary())

average treatment effect, E[Y^a] - E[Y^reference]
identified by explicit-adjustment: E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
adjustment/history: ['discharge_risk', 'prior_utilization', 'medication_burden', 'age']
required nuisances: ['outcome_regression', 'treatment_mechanism']
assumptions:
  - consistency: Y = Y^a when A = a
  - no interference: one unit's potential outcome does not depend on other units' treatment assignments
  - no unmeasured confounding: Y^a is independent of A given W
  - positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 2dd268e1f5ab29ae
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseline measu

**What this output tells you.** The first lines show the estimand and its observed-data formula.
The formula averages the outcome regression over all patients, once with the offer and once
without. The required nuisances are the outcome regression Q and the treatment mechanism g, the
probability of an offer given the covariates. The summary then lists four assumptions and repeats
the stored protocol.

| assumption | what it means for this program | can the data check it? |
| --- | --- | --- |
| consistency | an offer always means the declared bedside plan and two scheduled contacts | no |
| no interference | one patient's assignment does not change another patient's offer or outcome | no |
| no unmeasured confounding | the recorded baseline variables block every common cause of assignment and the score | no |
| positivity | each baseline profile has some chance of an offer and of usual support | partly, through the support report |

The [shared study design](index.md#the-shared-study-design) states how the program supports
consistency and no interference. This page changes nothing in it.

No unmeasured confounding needs a causal argument. For example, an unrecorded discharge-team
judgment that affects both assignment and recovery would violate it. No estimator repairs that
failure. The sensitivity step asks how strong such a judgment would need to be.

The synthetic law needs only four covariates. A real protocol should also evaluate pre-assignment
site, navigator-team, calendar, and language-access causes. Add them when the causal review places
them on a common-cause path.


## Step 6: estimate the ATE

The configuration is written out in full, so you see every choice.

| setting | value | what it does |
| --- | --- | --- |
| `outcome_learner` | gradient boosting | fits Q, the expected score given the offer and the covariates |
| `treatment_learner` | gradient boosting | fits g, the probability of an offer given the covariates |
| `CrossFitting(n_folds=5)` | five folds | predicts each row from models that did not see that row |
| `Inference(alpha=0.05)` | 95% interval | sets the interval level |
| `Runtime(random_state=21, n_jobs=1)` | fixed seed, one process | makes the fit reproducible |

Cross-fitting splits the rows into folds. Each nuisance prediction for a row comes from models fit
on the other folds. The [methods guide](../user-guide/methods-learners.md#two-fold-layers)
explains the two fold layers.

Targeting then runs once on the pooled out-of-fold predictions. It updates Q with a clever
covariate built from g, until the estimated efficient score equation is solved.
[Targeting and bounds](../user-guide/methods-learners.md#targeting-and-bounds) gives the options.


In [6]:
from cleverly import CrossFitting, Inference, ModelSpec, Runtime, TMLEMethod

flexible = TMLEMethod(
    models=ModelSpec(
        outcome_learner=HistGradientBoostingRegressor(random_state=21),
        treatment_learner=HistGradientBoostingClassifier(random_state=21),
    ),
    cross_fitting=CrossFitting(n_folds=5),
    inference=Inference(alpha=0.05),
    runtime=Runtime(random_state=21, n_jobs=1),
)
result = effect.estimate(method=flexible)

estimate = result["ate"]
print(result.summary())
print()
print(f"estimate:        {estimate.psi:.3f}")
print(f"standard error:  {estimate.std_error:.3f}")
print(f"95% CI:          ({estimate.ci[0]:.3f}, {estimate.ci[1]:.3f})")
print(f"population ATE:  {truth['ate']:.3f}")

Targeted maximum likelihood estimation
n = 3000; covariates = 4; P(A=1) = 0.4583
causal estimand: average treatment effect, E[Y^a] - E[Y^reference]
identification: explicit-adjustment; E_W[E(transition_score | transition_navigation=a, W)] - E_W[E(transition_score | transition_navigation=0, W)] for a in [1]
required nuisances: outcome_regression, treatment_mechanism
identification assumptions: consistency: Y = Y^a when A = a; no interference: one unit's potential outcome does not depend on other units' treatment assignments; no unmeasured confounding: Y^a is independent of A given W; positivity: P(transition_navigation = a | W) > 0 almost surely for every supported treatment level a in [0, 1]
causal study protocol: schema 1; 2dd268e1f5ab29ae
target population: Adults with a discharge-home order at a participating hospital during the enrollment period
eligibility: ['Age 18 years or older', 'Discharge home ordered at a participating hospital']
time zero: Discharge-home order, after baseli

**What this output tells you.** The summary repeats the estimand, the identification, and the
protocol with its fingerprint. It then reports the method configuration. It names the construction
as stacked CV-TMLE, with nuisances cross-fitted over 5 folds. It also shows the propensity
truncation bound, [0.0114, 0.9886].

The estimate is 1.809 with a standard error of 0.114. The 95% interval is (1.585, 2.032), and it
contains the true ATE of 1.750. That is one draw, not a coverage result.

The interval is built from the targeted influence curve, not from the outcome model's own
standard error. Its validity remains conditional on support, nuisance convergence, the product-rate
condition, and the declared dependence structure. The [cross-fitting tutorial](cross-fitting.ipynb)
shows why the fold separation matters for flexible learners.


## Step 7: which population is the number about?

The average treatment effect answers a question about every patient. A spread decision asks
something narrower.

| estimand | the question it answers | who asks it |
| --- | --- | --- |
| ATT | what did patients who received an offer gain from assignment? | the teams reviewing the rollout |
| ATE | what would the eligible population gain if everyone received an offer? | the program sponsor |
| ATC | what would patients who received usual support gain from an offer? | whoever is deciding on spread |

These are three parameters, not three estimates of one. A second law makes the gap visible, because
its effect modification is aligned with the propensity. Patients most likely to receive an offer
benefit most. The code fits all three with linear learners and prints each beside its true value.


In [7]:
from cleverly import ATC, ATT
from cleverly.datasets import make_heterogeneous

spread_frame, spread_truth = make_heterogeneous(n=3_000, seed=23)
spread_frame = spread_frame.rename(
    columns={
        "Y": "transition_score",
        "A": "transition_navigation",
        "W1": "discharge_risk",
        "W2": "medication_burden",
    }
)
spread_study = CausalStudy(
    spread_frame,
    design=PointTreatment(
        outcome="transition_score",
        treatment="transition_navigation",
        adjustment=("discharge_risk", "medication_burden"),
    ),
)
simple = TMLEMethod(
    models=ModelSpec(
        outcome_learner=LinearRegression(),
        treatment_learner=LogisticRegression(max_iter=1000),
    ),
    cross_fitting=CrossFitting(n_folds=5),
    runtime=Runtime(random_state=23, n_jobs=1),
)
for estimand, key in (
    (ATT(reference=0), "att"),
    (ATE(reference=0), "ate"),
    (ATC(reference=0), "atc"),
):
    point = spread_study.identify(estimand).estimate(method=simple)[key]
    low, high = point.ci
    print(
        f"{key}: {point.psi:6.3f}  CI=({low:.3f}, {high:.3f})  population {spread_truth[key]:.3f}"
    )

att:  1.718  CI=(1.562, 1.874)  population 1.691
ate:  1.015  CI=(0.896, 1.134)  population 1.000
atc:  0.338  CI=(0.160, 0.516)  population 0.309


**What this output tells you.** The three estimates are 1.718, 1.015, and 0.338. Their true values
are 1.691, 1.000, and 0.309. The population law has `att > ate > atc` by construction.

Its propensity is exactly logistic, so the treatment learner is correct. The linear outcome
learner omits the law's navigation-by-risk interaction, so these fits rest on the treatment model.
Do not use interval overlap as a test of the differences between these parameters.

Read that as a warning about spread. The offered patients' gain is the ATT. Patients who received
usual support would get the ATC, which here is a small fraction of it. A program that budgets the
network rollout against the ATT will overpromise.


## Step 8: the failure mode, both nuisance models misspecified

Double robustness means the point estimate stays consistent when *either* Q *or* g is consistent.
It is a claim about *or*, not about *and*. Use the known synthetic law to compare learner
combinations. Treat the result as an illustration, not as validation evidence.

A gradient-boosted learner can represent the law's nonlinear features. The linear learners omit
those features by construction. Three more fits show the resulting finite-sample pattern. Each line
prints the estimate, its interval, and its distance from the true ATE.


In [8]:
dr_points = {}


def fit(outcome_learner, treatment_learner, label):
    method = TMLEMethod(
        models=ModelSpec(outcome_learner=outcome_learner, treatment_learner=treatment_learner),
        cross_fitting=CrossFitting(n_folds=5),
        runtime=Runtime(random_state=21, n_jobs=1),
    )
    point = effect.estimate(method=method)["ate"]
    dr_points[label] = point
    low, high = point.ci
    miss = abs(point.psi - truth["ate"])
    print(f"{label:22s} psi={point.psi:6.3f}  CI=({low:.3f}, {high:.3f})  miss={miss:.3f}")


fit(
    HistGradientBoostingRegressor(random_state=21),
    LogisticRegression(max_iter=1000),
    "flexible Q, linear g",
)
fit(
    LinearRegression(),
    HistGradientBoostingClassifier(random_state=21),
    "linear Q, flexible g",
)
fit(LinearRegression(), LogisticRegression(max_iter=1000), "both linear")
print(f"population ATE: {truth['ate']:.3f}")

flexible Q, linear g   psi= 1.789  CI=(1.702, 1.876)  miss=0.039


linear Q, flexible g   psi= 1.764  CI=(1.414, 2.114)  miss=0.014
both linear            psi= 1.509  CI=(1.390, 1.628)  miss=0.241
population ATE: 1.750


**What this output tells you.** With one flexible learner, the fits miss the true ATE by 0.039 and
0.014. With both learners linear, the fit misses by 0.241, and its interval (1.390, 1.628) excludes
the true value of 1.750. On this draw, the fit with both linear learners misses the population value
by several times more than either fit with one flexible learner.

This output does not establish nuisance consistency or repeated-sampling coverage.

| caution | why |
| --- | --- |
| the linear models omit known terms | a real analysis does not reveal which nuisance model is consistent |
| one draw is not a coverage result | coverage is a repeated-sampling property |
| one nuisance is inconsistent | the point estimate can stay consistent, but the influence-curve interval need not be valid. [DR-TMLE](dr-tmle.ipynb) addresses that case |
| point consistency and interval validity differ | Wald inference needs the stated product-rate and regularity conditions |


## Step 9: diagnostics, what the fit can check

Start with the combined assessment of the Step 6 fit. It presents validation, diagnostics, and
sensitivity together. `assessment.attention` lists the failure and warning rows. The code then
prints three retained reports: support, nuisance models, and score equations.

The support report describes positivity, as the terms table defines it, in the fitted data. It shows how the fitted propensities
spread, how many rows hit the truncation bound, and how concentrated the weights are.


In [9]:
assessment = result.assess()
print(assessment.summary())
print("needs attention:", tuple(item.name for item in assessment.attention))

support = assessment.report("support")
nuisance = assessment.report("nuisance_models")
scores = assessment.report("score_equations")
print()
print(support.summary())
print()
print(nuisance.summary())
print()
print(scores.summary())

Returned results
----------------
surface      operation            result                                                                                                                                                                                 
-----------  -------------------  ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
validation   support              maximum truncated fraction 0.9%; minimum effective-sample-size ratio 20.1%; group load: mean:h1 276.5/3000 Kish-equivalent mask rows (9.2%; 9.2% all; draw 01 of 01); not estimator ESS
sensitivity  omitted_confounding  at the default strengths; cf_y=0.03, cf_d=0.03, rho=1; bias-adjusted interval [1.63, 1.988]                                                                                            
sensitivity  robustness_value     point robustness value 0.2638; confidence-limit value 0.2237

**What this output tells you.** Read the four parts in order.

| output part | what it shows on this draw |
| --- | --- |
| `Checks` and `needs attention` | one warning, `nuisance_models`. The score-equation check passed |
| positivity and overlap | 28 units (0.93%) sit at the truncation bound. The treated arm keeps an effective-sample-size ratio of 0.201 |
| nuisance model diagnostics | the boosted propensity is poorly calibrated, with a calibration slope of 0.47 against an ideal of 1 |
| score-equation check | targeting solved the estimated efficient score equation |

A `completed` row means the calculation ran. It is not a pass. Call `assessment.to_frame()` for the
complete row ledger.

Two lessons follow. First, a better-predicting assignment model is not automatically a better one
for this purpose. [Collaborative TMLE](collaborative-tmle.ipynb) chooses the assignment model by its
effect on the targeted estimate. Second, the report gives no positivity verdict, because no
universal cutoff applies. The support report describes fitted overlap and cannot verify population
positivity.


### The truncation curve

Truncation clips each fitted propensity into a bound, so no row gets an extreme weight. The curve
retargets the estimate at a range of bounds without refitting the nuisance models.


In [10]:
retargeted = result.assess(include_retargets=True)
curve = retargeted.report("truncation_curve")
columns = ["bound", "psi", "std_err", "ci_lower", "ci_upper", "truncated_fraction"]
print(curve[columns].round(4).to_string(index=False))

 bound    psi  std_err  ci_lower  ci_upper  truncated_fraction
0.0010 1.7929   0.1637    1.4722    2.1137              0.0000
0.0050 1.7924   0.1637    1.4716    2.1131              0.0017
0.0100 1.8068   0.1197    1.5723    2.0413              0.0060
0.0114 1.8089   0.1140    1.5854    2.0324              0.0093
0.0250 1.8161   0.0886    1.6425    1.9897              0.0240
0.0500 1.8164   0.0707    1.6777    1.9550              0.0527
0.1000 1.8003   0.0608    1.6811    1.9195              0.1130
0.2000 1.7906   0.0509    1.6909    1.8904              0.2760


**What this output tells you.** Each row is one bound. The fitted bound is 0.0114, where the
estimate is 1.8089. Across bounds from 0.0010 to 0.2000, the estimate stays between 1.7906 and
1.8164. The standard error falls from 0.1637 to 0.0509 as the bound clips more rows.

The largest bound clips a fraction of 0.2760 of the rows, so it changes the weight of more than a
quarter of the patients. The movement shows sensitivity to this regularization choice. Limited movement does not
verify positivity.


## Step 10: sensitivity, what the fit cannot check

Diagnostics cannot see unmeasured confounding. Sensitivity analysis asks how strong a confounder
would need to be to explain the result away. The assessment above already holds the robustness
value. The benchmark calibrates that strength against `discharge_risk`, the covariate behind the
recorded risk process.

The benchmark refits the nuisances once without `discharge_risk`. The call goes to the facade
because `assess(include_refits=True)` would also run the costlier refutations.


In [11]:
robustness = assessment.report("robustness_value")
benchmark = result.sensitivity.benchmark(covariates=("discharge_risk",))
bounds = result.sensitivity.omitted_confounding(cf_y=benchmark.cf_y, cf_d=benchmark.cf_d)
print(f"robustness value: {robustness['rv']:.3f} (confidence-limit value {robustness['rva']:.3f})")
print(benchmark)
low, high = bounds.ci_lower, bounds.ci_upper
print(f"bias-adjusted 95% CI at the benchmark strength: ({low:.3f}, {high:.3f})")

robustness value: 0.264 (confidence-limit value 0.224)
Benchmark for 'ate' against ['discharge_risk']
------------------------------------------------
quantity  with covariates  without
--------  ---------------  -------
estimate  1.8089           2.2662 
sigma^2   1.2359           2.1283 
nu^2      28.005           26.295 

implied cf_y = 0.7220, cf_d = 0.0650, rho = 0.3702
the estimate moved by +0.45728 when these covariates were dropped
bias-adjusted 95% CI at the benchmark strength: (0.199, 3.420)


**What this output tells you.** The robustness value is 0.264. It assumes worst-case alignment
(`rho=1`). It is the equal outcome-side and treatment-side strength that moves the point estimate
to zero.

The benchmark drops `discharge_risk` and refits. The estimate moves from 1.8089 to 2.2662. The
implied strengths are `cf_y = 0.7220` and `cf_d = 0.0650`. At those strengths and `rho=1`, the
bias-adjusted interval is (0.199, 3.420), which still excludes zero.

The review must decide whether an unrecorded team judgment could be stronger than the recorded risk
score. The
[omitted-variable bounds](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours)
section defines each quantity. `result.assess(include_refits=True)` adds placebo, noise, and
subsampling refutations. A stable refutation is not evidence of correctness.


## How far to trust this

The [stacked point-treatment CV-TMLE study](../technical-reference/method-evidence/stacked-point-treatment-cv-tmle.md)
validates this construction with GLM learners. No registered study covers the boosted learners
used here.

| layer | establishes | does not establish |
| --- | --- | --- |
| assessment overview | which stored checks need attention, and which operations did not run | the detail needed to interpret each retained report |
| retained diagnostics | that targeting converged, and how concentrated the fitted weights are | that the nuisance models are right |
| sensitivity analysis | how strong an unmeasured confounder would need to be | that no such confounder exists |
| the registered study | the implementation recovers known truths and behaves as its theory predicts | that your identification assumptions hold on your data |

Nothing in this list validates the causal reading. That rests on consistency, no interference, and
no unmeasured confounding. All three are arguments about the program rather than about the fit.


## Step 11: keep the result

A fit is an artifact. It carries the nuisance models, the influence curves, and the provenance
stamp, so the assessment replays without refitting. A program that reports quarterly needs that.
The code saves the result to a temporary directory, loads it, and checks the protocol fingerprint.


In [12]:
from pathlib import Path
from tempfile import TemporaryDirectory

from cleverly import load

with TemporaryDirectory() as directory:
    saved = Path(directory) / "transition-navigation-ate.joblib"
    result.save(saved)
    restored = load(saved)
    restored_protocol = restored.identified_effect.protocol
    assert restored_protocol is not None
    assert restored_protocol.fingerprint == restored.provenance.protocol_fingerprint
    print("restored protocol fingerprint:", restored_protocol.fingerprint)
    print(restored.replayability)
    replayed = restored.assess()
    print("needs attention after the restore:", tuple(item.name for item in replayed.attention))

restored protocol fingerprint: 2dd268e1f5ab29ae
Replayability(summarize_existing_artifacts=True, retarget_cached_nuisances=True, evaluate_stored_representer=False, refit_nuisances=True, evaluate_new_data=False, unreconstructible=())
needs attention after the restore: ('nuisance_models',)


**What this output tells you.** The restored protocol has the fingerprint `2dd268e1f5ab29ae`.
Step 4 printed the same fingerprint. The `replayability` attribute says which operations the restored
artifact can still perform. The replayed assessment names the same warning as Step 9.

A new nuisance refit still needs the analysis data. Use a maintained path for a real audit artifact.
Load only joblib files you trust, and keep the dependency versions compatible.


## Where to go next

This page treated discharges as independent rows. Read
[CV-TMLE and cross-fitting](cross-fitting.ipynb) for the same question at network scale, where patients
share navigator teams. If your worry is instead which baseline variables belong in the assignment
model, read [collaborative TMLE](collaborative-tmle.ipynb).

The [examples index](index.md#the-program) lists every tutorial in the program.
